<a href="https://colab.research.google.com/github/Grupo-IA-FIAP-EMLL-2026-2/fase5-cap1-farmtech/blob/main/EltonDias_rm572530_pbl_fase6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 6 — Entrega 2: YOLO padrão e comparação de abordagens

Este notebook compara a YOLOv5s customizada da Entrega 1, uma YOLO padrão pré-treinada em COCO e uma CNN MobileNetV3 treinada do zero. As classes são `banana` e `garrafa_pet`.

## Metodologia

A mesma separação de 64 imagens para treino, 8 para validação e 8 para teste é preservada. A YOLO customizada é avaliada por mAP; a CNN por acurácia; e a YOLO padrão por acerto de presença da classe na imagem. Como a YOLO COCO usa `bottle`, ela é uma baseline e não uma substituta direta da classe específica `garrafa_pet`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q gdown pyyaml pandas matplotlib

Mounted at /content/drive


In [ ]:
from pathlib import Path
import os

# Se esta pasta existir no Drive, os resultados serão gravados nela.
DRIVE_PROJECT = Path('/content/drive/MyDrive/FIAP/FASE-06')
LOCAL_OUTPUT = Path('/content/resultados_yolo_padrao')
OUTPUT_DIR = (DRIVE_PROJECT / 'entrega2_yolo_comparacao' / 'resultados_yolo_padrao'
              if DRIVE_PROJECT.exists() else LOCAL_OUTPUT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Resultados serão gravados em:', OUTPUT_DIR)

Resultados serão gravados em: /content/drive/MyDrive/FIAP/FASE-06/entrega2_yolo_comparacao/resultados_yolo_padrao


In [ ]:
# Baixa a pasta pública do dataset somente se ela não estiver no seu Drive.
import subprocess
import yaml
from pathlib import Path

possiveis = [DRIVE_PROJECT / 'dataset_yolo' / 'organizado', Path('/content/dataset_yolo/organizado')]
DATASET_ROOT = next((p for p in possiveis if (p / 'data.yaml').exists()), None)
if DATASET_ROOT is None:
    try:
        # Adicionada a flag --remaining-ok para mitigar problemas de limite de cota de download do gdown
        subprocess.run(['gdown', '--folder',
                        'https://drive.google.com/drive/folders/1stHR3rTiQuErFUsmGEW8zV0xdWDaiiaZ',
                        '--output', '/content/dataset_yolo', '--remaining-ok'], check=True)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            "Falha ao baixar o dataset via gdown. Isso geralmente ocorre devido a limites de cota temporários do Google Drive. "
            "Se o erro persistir, tente baixar a pasta manualmente e enviá-la para o seu Google Drive na pasta do projeto."
        ) from e

    candidatos = list(Path('/content/dataset_yolo').rglob('data.yaml'))
    if len(candidatos) != 1:
        raise RuntimeError(f'data.yaml não identificado: {candidatos}')
    DATASET_ROOT = candidatos[0].parent

with open(DATASET_ROOT / 'data.yaml', encoding='utf-8') as arquivo:
    nomes = yaml.safe_load(arquivo)['names']
imagens_teste = sorted(p for p in (DATASET_ROOT / 'images' / 'test').iterdir()
                       if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
if len(imagens_teste) != 8:
    raise RuntimeError(f'Esperadas 8 imagens de teste; encontradas {len(imagens_teste)}.')

def classe_real(imagem):
    caminho = DATASET_ROOT / 'labels' / 'test' / f'{imagem.stem}.txt'
    ids = {int(linha.split()[0]) for linha in caminho.read_text().splitlines() if linha.strip()}
    if len(ids) != 1:
        raise ValueError(f'{imagem.name}: era esperada uma única classe; encontrado {ids}')
    return nomes[ids.pop()]

print('Dataset:', DATASET_ROOT)
[(p.name, classe_real(p)) for p in imagens_teste]

Dataset: /content/drive/MyDrive/FIAP/FASE-06/dataset_yolo/organizado


[('20261009_191256.jpg', 'garrafa_pet'),
 ('20261011_220831.jpg', 'banana'),
 ('20261011_221012.jpg', 'banana'),
 ('IMG-20261005-WA0006.jpg', 'banana'),
 ('IMG-20261005-WA0008.jpg', 'banana'),
 ('IMG-20261005-WA0044.jpg', 'garrafa_pet'),
 ('IMG-20261005-WA0060.jpg', 'garrafa_pet'),
 ('IMG-20261005-WA0112.jpg', 'garrafa_pet')]

In [ ]:
# Instala a biblioteca ultralytics necessária para o carregamento do YOLOv5 via PyTorch Hub
%pip install -q ultralytics

# YOLO padrão: pesos COCO, sem treinamento com o dataset do grupo.
import torch
import time
import pandas as pd

modelo = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True, trust_repo=True, verbose=False)
modelo.conf = 0.25
modelo.iou = 0.60
modelo.classes = [39, 46]  # COCO: bottle e banana
COCO_PARA_PROJETO = {'banana': 'banana', 'bottle': 'garrafa_pet'}

# Aquecimento: não entra no tempo das imagens avaliadas.
_ = modelo(str(imagens_teste[0]), size=640)

linhas = []
for imagem in imagens_teste:
    inicio = time.perf_counter()
    resultado = modelo(str(imagem), size=640)
    tempo_ms = (time.perf_counter() - inicio) * 1000
    deteccoes = resultado.pandas().xyxy[0]
    deteccoes = deteccoes[deteccoes['name'].isin(COCO_PARA_PROJETO)]
    if deteccoes.empty:
        predicao, confianca = 'sem_deteccao', None
    else:
        melhor = deteccoes.sort_values('confidence', ascending=False).iloc[0]
        predicao = COCO_PARA_PROJETO[melhor['name']]
        confianca = float(melhor['confidence'])
    real = classe_real(imagem)
    linhas.append({'arquivo': imagem.name, 'classe_real': real,
                   'predicao_yolo_padrao': predicao, 'confianca': confianca,
                   'acertou': predicao == real, 'tempo_inferencia_ms': tempo_ms})

predicoes = pd.DataFrame(linhas)
predicoes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 704.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 5.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLOv5 🚀 2026-10-12 Python-3.13.16 torch-2.11.0+cpu CPU



100%|██████████| 14.1M/14.1M [00:00<00:00, 48.5MB/s]

Fusing layers... 
YOLOv5s summary: 124 layers, 7,225,885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


,arquivo,classe_real,predicao_yolo_padrao,confianca,acertou,tempo_inferencia_ms
0,20261009_191256.jpg,garrafa_pet,garrafa_pet,0.776662,True,440.143533
1,20261011_220831.jpg,banana,banana,0.870694,True,1461.059539
2,20261011_221012.jpg,banana,banana,0.676561,True,1208.247244
3,IMG-20261005-WA0006.jpg,banana,banana,0.668321,True,867.220796
4,IMG-20261005-WA0008.jpg,banana,banana,0.420469,True,874.444965
5,IMG-20261005-WA0044.jpg,garrafa_pet,sem_deteccao,NaN,False,908.815913
6,IMG-20261005-WA0060.jpg,garrafa_pet,sem_deteccao,NaN,False,887.130224
7,IMG-20261005-WA0112.jpg,garrafa_pet,sem_deteccao,NaN,False,852.177826


In [ ]:
# Persistência dos resultados efetivamente medidos nesta execução.
import json

predicoes.to_csv(OUTPUT_DIR / 'predicoes_yolo_padrao.csv', index=False)
resumo_padrao = {
    'modelo': 'YOLOv5s padrão pré-treinada no COCO',
    'regra_avaliacao': 'acerto por imagem; bottle do COCO aproxima garrafa_pet',
    'imagens_teste': int(len(predicoes)),
    'acertos': int(predicoes['acertou'].sum()),
    'acuracia_por_imagem': float(predicoes['acertou'].mean()),
    'tempo_medio_inferencia_ms': float(predicoes['tempo_inferencia_ms'].mean()),
    'limite_confianca': float(modelo.conf),
    'pesos': 'yolov5s COCO'
}
(OUTPUT_DIR / 'resultado_yolo_padrao.json').write_text(
    json.dumps(resumo_padrao, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(resumo_padrao, ensure_ascii=False, indent=2))

{
  "modelo": "YOLOv5s padrão pré-treinada no COCO",
  "regra_avaliacao": "acerto por imagem; bottle do COCO aproxima garrafa_pet",
  "imagens_teste": 8,
  "acertos": 5,
  "acuracia_por_imagem": 0.625,
  "tempo_medio_inferencia_ms": 937.4050050000164,
  "limite_confianca": 0.25,
  "pesos": "yolov5s COCO"
}


In [ ]:
# Comparação: métricas de detecção e classificação não são equivalentes.
comparacao = pd.DataFrame([
    {'abordagem': 'YOLOv5s customizada (60 épocas)', 'tipo': 'detecção',
     'métrica_principal': 'mAP@50', 'resultado_principal': 0.9000384337884338,
     'precisão': 0.9194824170922308, 'recall': 0.868421052631579,
     'mAP@50–95': 0.6616800180766798, 'tempo_treino_min': 2.6860117521999958,
     'tempo_inferencia_ms': None, 'observação': 'Resultados oficiais do grupo.'},
    {'abordagem': 'YOLOv5s padrão (COCO)', 'tipo': 'detecção',
     'métrica_principal': 'acurácia por imagem', 'resultado_principal': resumo_padrao['acuracia_por_imagem'],
     'precisão': None, 'recall': None, 'mAP@50–95': None, 'tempo_treino_min': 0.0,
     'tempo_inferencia_ms': resumo_padrao['tempo_medio_inferencia_ms'],
     'observação': 'Baseline sem ajuste; bottle é aproximação de garrafa_pet.'},
    {'abordagem': 'CNN do zero (Matheus)', 'tipo': 'classificação',
     'métrica_principal': 'acurácia no teste', 'resultado_principal': 1.0,
     'precisão': None, 'recall': None, 'mAP@50–95': None, 'tempo_treino_min': 15.6 / 60,
     'tempo_inferencia_ms': 9.4,
     'observação': 'Resultado reportado no notebook da branch matheus-fontes: 8/8.'}
])
comparacao.to_csv(OUTPUT_DIR / 'comparacao_abordagens.csv', index=False)
comparacao

,abordagem,tipo,métrica_principal,resultado_principal,precisão,recall,mAP@50–95,tempo_treino_min,tempo_inferencia_ms,observação
0,YOLOv5s customizada (60 épocas),detecção,mAP@50,0.900038,0.919482,0.868421,0.66168,2.686012,NaN,Resultados oficiais do grupo.
1,YOLOv5s padrão (COCO),detecção,acurácia por imagem,0.625000,NaN,NaN,NaN,0.000000,937.405005,Baseline sem ajuste; bottle é aproximação de g...
2,CNN do zero (Matheus),classificação,acurácia no teste,1.000000,NaN,NaN,NaN,0.260000,9.400000,Resultado reportado no notebook da branch math...


## Conclusão

- A **YOLO padrão** permite integração rápida, pois não exige treinamento, mas a classe `bottle` do COCO não é equivalente exatamente a `garrafa_pet`.
- A **YOLO customizada** é a mais indicada quando é necessário localizar o objeto na imagem; seu melhor experimento obteve mAP@50 de aproximadamente 90% na validação.
- A **CNN do zero** é adequada quando se quer apenas classificar a imagem. Ela não fornece caixas delimitadoras; portanto, sua acurácia não deve ser colocada no mesmo ranking do mAP da YOLO.

Antes de entregar, execute todas as células e envie este notebook já executado, junto dos três CSV/JSON criados em `resultados_yolo_padrao`.